In [ ]:
import pika
import json
from influxdb import InfluxDBClient
import datetime

credentials = pika.PlainCredentials('martin', 'martin00')
parameters =  pika.ConnectionParameters('149.62.71.186', credentials=credentials)
connection = pika.BlockingConnection(parameters)
channel = connection.channel()

host = '149.62.71.186'
user = 'admin'
password = 'fis_influx'
port=8086
client = InfluxDBClient(host, port, user, password)
client.create_database('Naloga 4')
client.switch_database('Naloga 4')

In [ ]:
result = channel.queue_declare('my_new_queue')
queue_name = result.method.queue

def callback(ch, method, properties, body):
    print("Prejeli smo %r" % (body.decode()))
    a = json.loads(body.decode())
    client.write_points([a])
    

channel.basic_consume(queue=queue_name, on_message_callback=callback, auto_ack=True)

channel.start_consuming()

In [ ]:
from influxdb import InfluxDBClient
import matplotlib.pyplot as plt

host = '149.62.71.186'
user = 'admin'
password = 'fis_influx'
port=8086
client = InfluxDBClient(host, port, user, password)


result=client.query('SELECT "I1" FROM "Naloga 4"."autogen"."Current"')
points = result.get_points()
times = []
values = []
for point in points:
    times.append(point['time'])
    values.append(point['I1'])
    
plt.plot(times, values, 'o', markersize = 2, color = 'blue')
plt.xticks(rotation=90, ha='right')  # 'ha' for horizontal alignment
plt.legend(['Plot 1', 'Plot 2'], fontsize = 16)

plt.xlabel('Čas ')
plt.ylabel('T1')